# Fig 3 — Representative naive vs aligned trace at offset = 20s

2 stacked panels showing one representative run per mode. Panel (a):
E2 naive — playhead jumps to live edge on switch. Panel (b): E3
aligned — playhead continues smoothly from the same media time.

In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "figures"))

import matplotlib.pyplot as plt

from _data import find_median_run_dir, load_run_metrics
from _style import (
    COL_ALIGNED,
    COL_NAIVE,
    COLUMN_WIDTH_IN,
    apply_acm_style,
)

apply_acm_style()

In [ ]:
naive_run = find_median_run_dir("e2", "naive_offset20", "max_playhead_gap_ms")
aligned_run = find_median_run_dir("e3", "aligned_offset20", "max_playhead_gap_ms")

naive_metrics = load_run_metrics(naive_run)
aligned_metrics = load_run_metrics(aligned_run)
naive_switches = json.loads((naive_run / "switch_records.json").read_text())
aligned_switches = json.loads((aligned_run / "switch_records.json").read_text())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(COLUMN_WIDTH_IN, 3.4),
                         sharex=True, constrained_layout=True)

def plot_panel(ax, metrics, switches, title, primary_color):
    # Left axis: playback position (current_time) vs wall clock.
    ax.plot(metrics["wall_clock_s"], metrics["current_time"],
            color=primary_color, linewidth=1.4, label="playback time (s)")
    ax.set_ylabel("playback time (s)")
    ax.set_title(title, loc="left", fontsize=8, pad=2)
    ax.set_ylim(bottom=0)

    # Right axis: bandwidth (kbps).
    ax2 = ax.twinx()
    ax2.plot(metrics["wall_clock_s"], metrics["bandwidth_bps"] / 1000,
             color="gray", linewidth=0.9, alpha=0.7, label="bandwidth (kbps)")
    ax2.set_ylabel("bw (kbps)", color="gray")
    ax2.tick_params(axis="y", labelcolor="gray")
    ax2.spines["top"].set_visible(False)

    # Mark each switch event.
    for s in [r for r in switches if r.get("eventType") == "switch"]:
        sent_s = s["switchSentAt"] / 1000.0
        ax.axvline(sent_s, color="black", linestyle=":", linewidth=0.7, alpha=0.5)
        gap_ms = s.get("playheadGapMs", 0)
        ax.annotate(
            f"playheadGap = {gap_ms:.0f} ms",
            xy=(sent_s, metrics["current_time"].max() * 0.5),
            xytext=(6, 0), textcoords="offset points",
            fontsize=6.5, va="center",
            arrowprops=dict(arrowstyle="-", lw=0.5, color="black"),
        )

plot_panel(axes[0], naive_metrics, naive_switches,
           "(a) Naive switch (E2, offset=20s)", COL_NAIVE)
plot_panel(axes[1], aligned_metrics, aligned_switches,
           "(b) Aligned switch (E3, offset=20s)", COL_ALIGNED)
axes[1].set_xlabel("wall-clock time (s)")

In [ ]:
fig.savefig(Path.cwd().parent / "figures" / "fig3_trace_naive_vs_aligned.pdf")
fig.savefig(Path.cwd().parent / "figures" / "fig3_trace_naive_vs_aligned.png", dpi=200)